<a href="https://colab.research.google.com/drive/1sJ_LL-f4VbVGMol2F-Y3XppIvnU1JXUB?usp=sharing" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<center>

<div style="text-align:center;">
<a href="https://lexsi-labs.github.io/CuratorKIT/latest/"><img src="https://raw.githubusercontent.com/Lexsi-Labs/CuratorKIT/refs/heads/main/docs/assets/logo.png" width="450"></a>

<a href="https://lexsi-labs.github.io/CuratorKIT/latest/"><img src="https://raw.githubusercontent.com/Lexsi-Labs/TabTune/refs/heads/docs/assets/docs.png" width="150"></a>

</div>

<i>Star us on <a href="https://github.com/Lexsi-Labs/CuratorKIT">Github</a> </i>

To run this, press "*Runtime*" and press "*Run all*" on a Google Colab instance!

<div style="text-align:center;">
<a href="https://arxiv.org/abs/2606.21631"><img src="https://raw.githubusercontent.com/Lexsi-Labs/TabTune/refs/heads/docs/assets/lexsilogo.png" width="200"></a>
</div>
</center>




# Filtered vs unfiltered fine-tuning

**Gates:** `HallucinationGate + RewardGate` &nbsp;|&nbsp; **Judge:** Qwen3.6-35B-A3B-FP8 &nbsp;|&nbsp; **Trainer:** Unsloth + LoRA on Qwen3-1.7B

End-to-end case study: fine-tune the same base model twice, once on the samples that passed, once on those that were rejected.

**Metrics**

| Metric | What it measures |
|---|---|
| ROUGE-L | Lexical overlap with the reference answer |
| BERTScore-F1 | Semantic similarity to the reference answer |
| Faithfulness | BERTScore F1 between prediction and the source chunk (higher = more grounded) |

## 1. Setup

**Runtime:** A100 (or any GPU ≥ 16 GB VRAM) + High-RAM. Run the install cell, then **Runtime --> Restart session** once before continuing.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# 0. Setup — clone + install CuratorKIT (run once, then comment out)
# ═══════════════════════════════════════════════════════════════════════════
from pathlib import Path

repo_dir = Path.home() / "CuratorKIT"
if not repo_dir.exists():
    !git clone https://github.com/Lexsi-Labs/CuratorKIT.git {repo_dir}

!pip install -e "{repo_dir}[generation]" -q

# Extra deps for fine-tuning + eval
!pip install -q "numpy<2" "pandas>=2.0,<2.3"
!pip install -q datasets evaluate openai litellm tenacity nest-asyncio
!pip install -q rouge-score bert-score
!pip install -q "unsloth[colab-new]" "trl<0.10" peft accelerate bitsandbytes

print(f"CuratorKIT installed from {repo_dir}")

## 2. Configuration

**Judge endpoint**: we use a hosted Qwen3.6-35B in non-thinking mode. To swap in your own:

**vLLM** (recommended for production):
```bash
vllm serve Qwen/Qwen3.6-35B-A3B-FP8 --port 8000 --enable-chunked-prefill
# Then set: JUDGE_API_BASE = "http://localhost:8000/v1"
```

**Ollama** (lighter, for local laptops):
```bash
ollama serve
ollama pull qwen3:32b
# Then set: JUDGE_API_BASE = "http://localhost:11434/v1", JUDGE_MODEL = "ollama/qwen3:32b"
```

In [ ]:
from pathlib import Path

# ── Source corpus (raw Qwen3-4B QA, pulled from HF dataset) ──────────
HF_DATASET    = "Lexsi/provenance-grounded-synthetic-qa"
HF_DATA_FILE  = "train_unfiltered_qa/qwen3_4b/train.jsonl"
TEST_SIZE     = 500    # held-out from the loaded corpus
SPLIT_SEED    = 42

# ── Judge model (used by both gates) ─────────────────────────────────
JUDGE_MODEL    = "openai/Qwen/Qwen3.6-35B-A3B-fp8"   # litellm-style: provider/model
JUDGE_API_BASE = " "
JUDGE_API_KEY  = "..."   # set your endpoint key

# Gates ─ thresholds match the CuratorKIT paper
HALL_THRESHOLD   = 0.7
REWARD_THRESHOLD = 0.7
JUDGE_CONCURRENCY = 24

# ── Fine-tuning (Qwen3-1.7B, 4-bit Unsloth + LoRA) ────────────────────
BASE_MODEL  = "unsloth/Qwen3-1.7B-bnb-4bit"
MAX_SEQ_LEN = 2048
LORA_R, LORA_ALPHA = 16, 32
EPOCHS, BATCH_SIZE, GRAD_ACCUM = 2, 4, 4
POOL_SIZE   = 1100           # samples per condition (filtered / unfiltered, disjoint)
EVAL_SAMPLES = 300           # subset of the test set to score

OUTPUT_DIR = Path("output/cuad_filtered_vs_unfiltered")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Judge        : {JUDGE_MODEL}")
print(f"Gates        : Hallucination(thr={HALL_THRESHOLD}) + Reward(thr={REWARD_THRESHOLD})")
print(f"Base model   : {BASE_MODEL}")
print(f"Pool size    : {POOL_SIZE} samples per condition (filtered vs unfiltered, disjoint)")
print(f"Test size    : {TEST_SIZE} held out (eval on {EVAL_SAMPLES})")

## 3. Load `train.jsonl`

In [ ]:
from datasets import load_dataset, Dataset
import pandas as pd, json, random

raw_ds = load_dataset(HF_DATASET, data_files=HF_DATA_FILE, split="train")
raw_rows = list(raw_ds)
print(f"Loaded {len(raw_rows):,} rows from HF: {HF_DATASET} / {HF_DATA_FILE}")

random.seed(SPLIT_SEED)
shuffled = raw_rows[:]; random.shuffle(shuffled)
test_rows  = shuffled[:TEST_SIZE]
pool_rows  = shuffled[TEST_SIZE:]

test_set      = Dataset.from_list(test_rows)
unfiltered_ds = Dataset.from_list(pool_rows)

# Materialise the working pool as JSONL for Curator.
unfiltered_jsonl = OUTPUT_DIR / "unfiltered_corpus.jsonl"
with open(unfiltered_jsonl, "w") as f:
    for r in unfiltered_ds:
        f.write(json.dumps({
            "id":          r["id"],
            "source_uri":  r.get("source_uri", ""),
            "instruction": r["instruction"],
            "input":       r["input"] or r.get("source_chunk", ""),
            "output":      r["output"],
            "task_type":   "instruction_following",
            "metadata":    {"domain": r.get("domain"), "source_chunk": r.get("source_chunk", "")},
        }) + "\n")

print(f"Working pool : {len(unfiltered_ds):,} samples → {unfiltered_jsonl}")
print(f"Test set     : {len(test_set):,} samples (held-out)")
print(f"Pool domains : {dict(pd.Series(unfiltered_ds['domain']).value_counts())}")
print(f"Pool adv. inj: {sum(unfiltered_ds['injected'])} / {len(unfiltered_ds)}")

## 4. Curation with `CuratorConfig`


In [8]:
from curatorkit import Curator, CuratorConfig
from types import SimpleNamespace
import nest_asyncio, json, gc, os, litellm
nest_asyncio.apply()

# ── Cap LiteLLM's unbounded global retention ────────────────────────
litellm.success_callback = []
litellm.failure_callback = []
litellm._async_success_callback = []
litellm._async_failure_callback = []
litellm.callbacks = []
os.environ["LITELLM_LOG"] = "ERROR"
os.environ["LITELLM_DROP_PARAMS"] = "True"
try:
    litellm.suppress_debug_info = True
except Exception:
    pass

CHUNK_SIZE = 500   # process the working pool in chunks

def make_config(src_jsonl, out_subdir):
    return CuratorConfig(
        dataset=str(src_jsonl), format="alpaca",   # schema = instruction/input/output
        schema_gate=True, min_tokens=10, max_tokens=2048,
        dedup="exact", clean=True,
        llm_model=JUDGE_MODEL, llm_api_base=JUDGE_API_BASE, llm_api_key=JUDGE_API_KEY,
        llm_extra_body={"chat_template_kwargs": {"enable_thinking": False}},
        llm_concurrency=JUDGE_CONCURRENCY,
        judge_llm_extra_body={"chat_template_kwargs": {"enable_thinking": False}},
        judge_concurrency=JUDGE_CONCURRENCY,
        hallucination_threshold=HALL_THRESHOLD,
        reward_threshold=REWARD_THRESHOLD,
        reward_dimensions=["helpfulness", "honesty", "instruction_following", "depth"],
        output_dir=str(OUTPUT_DIR / out_subdir),
        export_formats=["alpaca"],
        name=f"chunk_{out_subdir}",
    )

# Split unfiltered_jsonl into chunks on disk
chunk_dir = OUTPUT_DIR / "chunks"; chunk_dir.mkdir(exist_ok=True)
with open(unfiltered_jsonl) as f:
    all_lines = f.readlines()
chunk_paths = []
for i in range(0, len(all_lines), CHUNK_SIZE):
    cp = chunk_dir / f"chunk_{i//CHUNK_SIZE:03d}.jsonl"
    cp.write_text("".join(all_lines[i:i+CHUNK_SIZE]))
    chunk_paths.append(cp)
print(f"Split {len(all_lines):,} samples into {len(chunk_paths)} chunks of {CHUNK_SIZE}\n")

all_passed, all_rejected = [], []
for idx, cp in enumerate(chunk_paths, 1):
    print(f"══ Chunk {idx}/{len(chunk_paths)} ({cp.name}) ══")
    res = Curator(make_config(cp, f"curator_run/chunk_{idx:03d}")).run()
    all_passed.extend(res.passed)
    all_rejected.extend(res.rejected)
    del res
    litellm.success_callback.clear()
    if hasattr(litellm, "_async_success_callback"):
        litellm._async_success_callback.clear()
    gc.collect()
    print(f"   running totals: passed={len(all_passed):,}  rejected={len(all_rejected):,}\n")

result = SimpleNamespace(
    passed=all_passed, rejected=all_rejected,
    print_summary=lambda: print(f"TOTAL  passed={len(all_passed):,}  rejected={len(all_rejected):,}"),
)
result.print_summary()

Split 4,500 samples into 9 chunks of 500

══ Chunk 1/9 (chunk_000.jsonl) ══


RewardGate: 100%|██████████| 339/339 [02:38<00:00,  2.14sample/s, passed=171, rejected=168]


   running totals: passed=171  rejected=329

══ Chunk 2/9 (chunk_001.jsonl) ══


RewardGate: 100%|██████████| 321/321 [02:28<00:00,  2.17sample/s, passed=188, rejected=133]


   running totals: passed=359  rejected=641

══ Chunk 3/9 (chunk_002.jsonl) ══


...

   running totals: passed=1,371  rejected=2,129

══ Chunk 8/9 (chunk_007.jsonl) ══


RewardGate: 100%|██████████| 343/343 [02:38<00:00,  2.17sample/s, passed=182, rejected=161]


   running totals: passed=1,553  rejected=2,447

══ Chunk 9/9 (chunk_008.jsonl) ══


RewardGate: 100%|██████████| 344/344 [02:37<00:00,  2.19sample/s, passed=187, rejected=157]


   running totals: passed=1,740  rejected=2,760

TOTAL  passed=1,740  rejected=2,760


## 5. Inspect rejection breakdown

Bucket rejections by which step caught them. 

In [32]:
from collections import Counter
import textwrap, random

by_step = Counter(r.rejecting_step for r in result.rejected)
print(f"Total rejected: {len(result.rejected):,} / {len(result.passed) + len(result.rejected):,}\n")
print("Rejections by step:")
for step, n in sorted(by_step.items(), key=lambda x: -x[1]):
    pct = n / len(result.rejected) if result.rejected else 0
    print(f"  {step:<25} {n:>5,}  ({pct:.1%})")

def show(step):
    rs = [r for r in result.rejected if r.rejecting_step == step]
    if not rs: return
    random.seed(42)
    s = random.choice(rs)
    print(f"\n━━━ example rejected by {step} ━━━")
    print(f"  reason : {s.rejection_reason}")
    print(f"  Q      : {textwrap.shorten(s.instruction, 180)}")
    print(f"  A      : {textwrap.shorten(s.output, 240)}")
    print(f"  source : {textwrap.shorten(s.input, 180)}")

show("HallucinationGate")
show("RewardGate")

Total rejected: 2,760 / 4,500

Rejections by step:
  RewardGate                1,400  (50.7%)
  HallucinationGate         1,359  (49.2%)
  SchemaGate                    1  (0.0%)

━━━ example rejected by HallucinationGate ━━━
  reason : hallucination_contract_failed:0.40
  Q      : What was the purpose of the telephone interviews?
  A      : The telephone interviews aimed to gather additional information on health outcomes and related factors after the initial hospital discharge.
  source : Data were collected as part of a cross-sectional survey of postpartum health outcomes, service use, and costs of care in the first 4 weeks after postpartum hospital [...]

━━━ example rejected by RewardGate ━━━
  reason : below_reward_threshold:0.55
  Q      : What additional benefits did some studies identify for dexamethasone?
  A      : Some studies found potential additional benefits of dexamethasone, including improved compliance and less vomiting. These findings suggest that dexamethasone may 

## 6. Build the two matched pools — disjoint, equal size

The gates split the working pool into `passed` and `rejected`. We sample `POOL_SIZE` from each, so the two training sets are the same size and share no samples:

- **`filtered`** : `POOL_SIZE` rows drawn from `result.passed` (survived `HallucinationGate + RewardGate`).
- **`unfiltered`** : `POOL_SIZE` rows drawn from `result.rejected` (the samples the gates would have removed).


In [33]:
import random

def sample_to_dict(s):
    return {"instruction": s.instruction, "input": s.input, "output": s.output}

filtered_pool_full   = [sample_to_dict(s) for s in result.passed]
unfiltered_pool_full = [sample_to_dict(s) for s in result.rejected]

# Disjoint by construction (passed ∩ rejected = ∅). Equal size.
n_take = min(POOL_SIZE, len(filtered_pool_full), len(unfiltered_pool_full))

random.seed(42); train_filtered   = random.sample(filtered_pool_full,   n_take)
random.seed(42); train_unfiltered = random.sample(unfiltered_pool_full, n_take)

# Sanity: zero overlap between the two training sets.
filtered_keys   = {(r["instruction"], r["output"]) for r in train_filtered}
unfiltered_keys = {(r["instruction"], r["output"]) for r in train_unfiltered}
assert not (filtered_keys & unfiltered_keys), "filtered and unfiltered pools overlap"

TRAINING_SETS = {"unfiltered": train_unfiltered, "filtered": train_filtered}
print(f"Pool sizes : filtered={len(filtered_pool_full):,}  rejected={len(unfiltered_pool_full):,}")
print(f"Taking n_take = {n_take} per condition (disjoint).\n")
for name, ts in TRAINING_SETS.items():
    print(f"  {name:<10} : {len(ts):,} samples")

Pool sizes : filtered=1,740  rejected=2,760
Taking n_take = 1100 per condition (disjoint).

  unfiltered : 1,100 samples
  filtered   : 1,100 samples


## 7. Fine-tuning

Same base model (Qwen3-1.7B 4-bit), same hyperparameters, same `POOL_SIZE`. Only the source pool changes : `passed` (filtered) vs `rejected` (unfiltered).

In [34]:
from unsloth import FastLanguageModel
from datasets import Dataset
from trl import SFTTrainer
from transformers import TrainingArguments
import torch, gc

SYSTEM_PROMPT = "You are a helpful assistant that answers questions accurately based on the provided context."

def format_for_sft(record, tokenizer):
    instruction = (record.get("instruction") or "").strip()
    context     = (record.get("input")       or "").strip()
    answer      = (record.get("output")      or "").strip()
    user_content = instruction + (f"\n\nContext:\n{context}" if context else "")
    messages = [
        {"role": "system",    "content": SYSTEM_PROMPT},
        {"role": "user",      "content": user_content},
        {"role": "assistant", "content": answer},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)

def train_one(condition, records, out_dir):
    print(f"\n══════ Fine-tuning on `{condition}` ({len(records):,} samples) ══════")
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=BASE_MODEL, max_seq_length=MAX_SEQ_LEN, dtype=None, load_in_4bit=True,
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    model = FastLanguageModel.get_peft_model(
        model, r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=0.05,
        target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
        bias="none", use_gradient_checkpointing="unsloth", random_state=42,
    )

    texts = [format_for_sft(r, tokenizer) for r in records]
    train_ds = Dataset.from_dict({"text": texts})

    trainer = SFTTrainer(
        model=model, tokenizer=tokenizer, train_dataset=train_ds,
        dataset_text_field="text", max_seq_length=MAX_SEQ_LEN, packing=False,
        args=TrainingArguments(
            output_dir=str(out_dir), num_train_epochs=EPOCHS,
            per_device_train_batch_size=BATCH_SIZE, gradient_accumulation_steps=GRAD_ACCUM,
            learning_rate=2e-4, warmup_ratio=0.05, lr_scheduler_type="cosine",
            logging_steps=20, save_strategy="no",
            bf16=torch.cuda.is_bf16_supported(),
            fp16=not torch.cuda.is_bf16_supported(),
            gradient_checkpointing=True, seed=42, report_to="none",
        ),
    )
    trainer.train()
    model.save_pretrained(str(out_dir))
    tokenizer.save_pretrained(str(out_dir))
    del model, tokenizer, trainer; gc.collect(); torch.cuda.empty_cache()
    return out_dir

adapter_paths = {
    cond: train_one(cond, recs, OUTPUT_DIR / f"adapter_{cond}")
    for cond, recs in TRAINING_SETS.items()
}
print("\nAdapters saved:", {k: str(v) for k, v in adapter_paths.items()})


══════ Fine-tuning on `unfiltered` (1,100 samples) ══════
==((====))==  Unsloth 2025.10.8: Fast Qwen3 patching. Transformers: 4.56.2.
   \\   /|    NVIDIA RTX PRO 6000 Blackwell Server Edition. Num GPUs = 1. Max memory: 94.971 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


/content/unsloth_compiled_cache/UnslothSFTTrainer.py:674: UserWarning: You passed a `max_seq_length` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(
/content/unsloth_compiled_cache/UnslothSFTTrainer.py:712: UserWarning: You passed a `dataset_text_field` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(


Map:   0%|          | 0/1100 [00:00<?, ? examples/s]

/content/unsloth_compiled_cache/UnslothSFTTrainer.py:807: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `UnslothSFTTrainer.__init__`. Use `processing_class` instead.
  super().__init__(
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,100 | Num Epochs = 2 | Total steps = 138
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 17,432,576 of 1,738,007,552 (1.00% trained)


Step,Training Loss
20,2.441900
40,1.974700
60,1.934500
80,1.842900
100,1.831000
120,1.822900


...

/content/unsloth_compiled_cache/UnslothSFTTrainer.py:674: UserWarning: You passed a `max_seq_length` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(
/content/unsloth_compiled_cache/UnslothSFTTrainer.py:712: UserWarning: You passed a `dataset_text_field` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(


Map:   0%|          | 0/1100 [00:00<?, ? examples/s]

/content/unsloth_compiled_cache/UnslothSFTTrainer.py:807: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `UnslothSFTTrainer.__init__`. Use `processing_class` instead.
  super().__init__(
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,100 | Num Epochs = 2 | Total steps = 138
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 17,432,576 of 1,738,007,552 (1.00% trained)


Step,Training Loss
20,2.238400
40,1.796900
60,1.729100
80,1.654600
100,1.632800
120,1.616500



Adapters saved: {'unfiltered': 'output/cuad_filtered_vs_unfiltered/adapter_unfiltered', 'filtered': 'output/cuad_filtered_vs_unfiltered/adapter_filtered'}


## 8. Generate predictions on the held-out test set



In [35]:
from unsloth import FastLanguageModel
from transformers import AutoTokenizer
from tqdm.auto import tqdm
import torch, gc

eval_subset = test_set.shuffle(seed=42).select(range(min(EVAL_SAMPLES, len(test_set))))

def build_messages(record):
    instruction = (record.get("instruction") or "").strip()
    context     = (record.get("input")       or "").strip()
    user_content = instruction + (f"\n\nContext:\n{context}" if context else "")
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": user_content},
    ]

def generate_for(adapter_path):
    # Load model via Unsloth (gets the LoRA adapter merged for inference).
    model, _unsloth_tok = FastLanguageModel.from_pretrained(
        model_name=str(adapter_path), max_seq_length=MAX_SEQ_LEN, dtype=None, load_in_4bit=True,
    )
    FastLanguageModel.for_inference(model)

    # Load the tokenizer FRESH from the same path with padding_side="left" set
    # at construction time. This is the only place transformers honors it
    # without ambiguity (HF docs: huggingface.co/docs/transformers/llm_tutorial).
    tokenizer = AutoTokenizer.from_pretrained(str(adapter_path), padding_side="left")
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # Render chat-templated prompts (strings).
    prompts = [
        tokenizer.apply_chat_template(build_messages(ex), tokenize=False, add_generation_prompt=True)
        for ex in eval_subset
    ]

    outs, BS = [], 8
    for i in tqdm(range(0, len(prompts), BS), desc=adapter_path.name):
        batch = prompts[i:i+BS]
        enc = tokenizer(
            batch, return_tensors="pt", padding=True, truncation=True,
            max_length=MAX_SEQ_LEN - 256,
        ).to("cuda")
        with torch.no_grad():
            gen = model.generate(
                **enc, max_new_tokens=256, do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
            )
        # Left-padded → all prompts end at column enc["input_ids"].shape[1].
        prompt_len = enc["input_ids"].shape[1]
        for j in range(len(batch)):
            text = tokenizer.decode(gen[j][prompt_len:], skip_special_tokens=True)
            outs.append(text.strip())

    del model, tokenizer, _unsloth_tok; gc.collect(); torch.cuda.empty_cache()
    return outs

predictions = {cond: generate_for(p) for cond, p in adapter_paths.items()}
print({cond: len(p) for cond, p in predictions.items()})

==((====))==  Unsloth 2025.10.8: Fast Qwen3 patching. Transformers: 4.56.2.
   \\   /|    NVIDIA RTX PRO 6000 Blackwell Server Edition. Num GPUs = 1. Max memory: 94.971 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


adapter_unfiltered:   0%|          | 0/38 [00:00<?, ?it/s]

==((====))==  Unsloth 2025.10.8: Fast Qwen3 patching. Transformers: 4.56.2.
   \\   /|    NVIDIA RTX PRO 6000 Blackwell Server Edition. Num GPUs = 1. Max memory: 94.971 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


adapter_filtered:   0%|          | 0/38 [00:00<?, ?it/s]

{'unfiltered': 300, 'filtered': 300}


## 9. Score: ROUGE-L, BERTScore, Faithfulness


- **ROUGE-L** — lexical overlap with the reference answer.
- **BERTScore-F1** — semantic similarity to the reference answer.
- **Faithfulness** — BERTScore F1 between the prediction and the source chunk (the test set's `input` field). Higher = more grounded in source.

In [36]:
import evaluate
import pandas as pd

rouge_metric     = evaluate.load("rouge")
bertscore_metric = evaluate.load("bertscore")

references    = [ex["output"] for ex in eval_subset]
source_chunks = [ex.get("input") or ex.get("output") or "" for ex in eval_subset]

results = []
for cond, preds in predictions.items():
    rouge_res = rouge_metric.compute(predictions=preds, references=references, use_stemmer=True)
    bs_ref    = bertscore_metric.compute(predictions=preds, references=references,
                                         lang="en", batch_size=32)
    bs_faith  = bertscore_metric.compute(predictions=preds, references=source_chunks,
                                         lang="en", batch_size=32)
    results.append({
        "condition":    cond,
        "n_train":      len(TRAINING_SETS[cond]),
        "ROUGE-L":      round(rouge_res["rougeL"], 4),
        "BERTScore-F1": round(sum(bs_ref["f1"]) / len(bs_ref["f1"]), 4),
        "Faithfulness": round(sum(bs_faith["f1"]) / len(bs_faith["f1"]), 4),
    })
    print(results[-1])

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


{'condition': 'unfiltered', 'n_train': 1100, 'ROUGE-L': 0.4402, 'BERTScore-F1': 0.8893, 'Faithfulness': 0.8412}
{'condition': 'filtered', 'n_train': 1100, 'ROUGE-L': 0.4307, 'BERTScore-F1': 0.8964, 'Faithfulness': 0.8545}


## 10. Results

In [37]:
df = pd.DataFrame(results).set_index("condition")
base = df.loc["unfiltered"]
df["ΔROUGE-L"]      = (df["ROUGE-L"]      - base["ROUGE-L"]).round(4)
df["ΔBERTScore-F1"] = (df["BERTScore-F1"] - base["BERTScore-F1"]).round(4)
df["ΔFaithfulness"] = (df["Faithfulness"] - base["Faithfulness"]).round(4)
df

,n_train,ROUGE-L,BERTScore-F1,Faithfulness,ΔROUGE-L,ΔBERTScore-F1,ΔFaithfulness
condition,,,,,,,
unfiltered,1100,0.4402,0.8893,0.8412,0.0000,0.0000,0.0000
filtered,1100,0.4307,0.8964,0.8545,-0.0095,0.0071,0.0133
